# Module 3: LLM APIs & Application Development
**Codomax Gen AI Internship — Day 11 to Day 15**

This notebook covers everything you need to know about working with LLM APIs in Python, from authentication to building a full-featured AI chat system.

### What you will learn:
1. **LLM API Authentication** — API keys, environment variables, `.env` files
2. **Connecting to LLM APIs** — Google Gemini & OpenAI
3. **Conversation Management** — System, user, and assistant message roles
4. **Chat History** — Multi-turn conversations with context
5. **Model Parameters** — Temperature, max_tokens, top_p optimization
6. **Structured Responses** — Forcing JSON output, parsing structured data
7. **Error Handling** — Retry logic, rate limits, graceful degradation
8. **Cost Management** — Token counting and API cost estimation

---
## Section 1: Setup & Installation

Install the required packages. In Google Colab, these install in seconds.

In [ ]:
# Install required packages
!pip install google-generativeai python-dotenv openai tiktoken rich --quiet
print("Packages installed successfully!")

In [ ]:
# Import core libraries
import os
import json
import time
import re
from datetime import datetime
from typing import Optional, List, Dict, Any

# LLM SDK imports
import google.generativeai as genai

print("Libraries imported successfully!")
print(f"Notebook run date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

---
## Section 2: API Authentication & Environment Variables

### Why use environment variables?

**Never hardcode API keys in your code!** If you push code to GitHub with a hardcoded key, it can be stolen within minutes.

**Best practices:**
- Store keys in a `.env` file (never commit this to git)
- Use `os.getenv()` to read keys at runtime
- In Google Colab, use **Secrets** (the lock icon in the left panel)

In [ ]:
# Method 1: Google Colab Secrets (recommended for Colab)
try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
    print("API key loaded from Colab Secrets")
except Exception:
    # Method 2: Environment variable / .env file
    from dotenv import load_dotenv
    load_dotenv()
    GEMINI_API_KEY = os.getenv("GEMINI_API_KEY", "")
    if GEMINI_API_KEY:
        print("API key loaded from environment variable")
    else:
        # Method 3: Manual input (fallback)
        import getpass
        GEMINI_API_KEY = getpass.getpass("Enter your Gemini API key: ")
        print("API key entered manually")

# Validate key format (Gemini keys start with AIza)
if GEMINI_API_KEY and GEMINI_API_KEY.startswith("AIza"):
    print(f"Key format looks valid: {GEMINI_API_KEY[:8]}...{GEMINI_API_KEY[-4:]}")
else:
    print("Warning: Key may be invalid - check it starts with AIza")

In [ ]:
# Configure the Gemini SDK with our API key
genai.configure(api_key=GEMINI_API_KEY)
print("Gemini SDK configured!")

# List available models
print("\nAvailable Gemini models:")
for m in genai.list_models():
    if 'generateContent' in m.supported_generation_methods:
        print(f"  - {m.name}")

---
## Section 3: Making Your First API Call

Let's make our very first call to the Gemini API!

In [ ]:
# Create a model instance
model = genai.GenerativeModel("gemini-1.5-flash")

# Make a simple single-turn API call
print("Sending request to Gemini...")
response = model.generate_content("What is artificial intelligence in one sentence?")

print("Response received!")
print("-" * 50)
print(response.text)

In [ ]:
# Exploring the response object
print("Response Object Details:")
print(f"  Type: {type(response)}")
print(f"  Text preview: {response.text[:100]}")
print(f"  Finish reason: {response.candidates[0].finish_reason}")

# Token usage
usage = response.usage_metadata
print("\nToken Usage:")
print(f"  Prompt tokens    : {usage.prompt_token_count}")
print(f"  Completion tokens: {usage.candidates_token_count}")
print(f"  Total tokens     : {usage.total_token_count}")

---
## Section 4: Message Roles — System, User & Assistant

Every LLM conversation uses three types of messages:

| Role | Purpose | Example |
|------|---------|---------|
| `system` | Sets AI behaviour & persona | "You are a Python expert" |
| `user` | Human's message | "Explain decorators" |
| `model`/`assistant` | AI's reply | "A decorator is..." |

In Gemini, the assistant role is called `"model"`.

In [ ]:
# Using a system instruction to set the AI's persona
model_with_system = genai.GenerativeModel(
    model_name="gemini-1.5-flash",
    system_instruction=(
        "You are a Python expert who always provides code examples. "
        "Keep explanations concise and beginner-friendly."
    )
)

response = model_with_system.generate_content("What is a list comprehension?")
print("Response with system instruction:")
print(response.text)

In [ ]:
# Comparing different system instructions — same question, different personas
personas = {
    "Enthusiastic Teacher": "You are an enthusiastic teacher who uses emojis and real-world analogies.",
    "Technical Expert": "You are a senior software engineer. Be technical and precise.",
    "Poet": "You are a creative poet. Explain everything in rhyming verse.",
}

question = "What is machine learning?"

for persona_name, system_prompt in personas.items():
    model_persona = genai.GenerativeModel(
        model_name="gemini-1.5-flash",
        system_instruction=system_prompt
    )
    response = model_persona.generate_content(question)
    print(f"\n{'='*60}")
    print(f"Persona: {persona_name}")
    print('='*60)
    print(response.text[:400])

---
## Section 5: Multi-Turn Conversations & Chat History

Real chatbots need to remember what was said earlier.

In [ ]:
# Starting a chat session with history
chat = model.start_chat(history=[])

print("Starting a multi-turn conversation...")
print("=" * 60)

def send_message(chat_session, user_message, show_tokens=True):
    """Helper: send a message and print the response."""
    print(f"\nUSER: {user_message}")
    response = chat_session.send_message(user_message)
    print(f"AI  : {response.text}")
    if show_tokens:
        usage = response.usage_metadata
        print(f"   [prompt tokens: {usage.prompt_token_count}, output tokens: {usage.candidates_token_count}]")
    return response

# Turn 1
send_message(chat, "My name is Alex. I'm learning Python.")

# Turn 2 — the AI should remember the name
send_message(chat, "What should I learn next?")

# Turn 3 — building on context
send_message(chat, "Can you give me a small project idea based on what we discussed?")

In [ ]:
# Inspecting the full conversation history
print("\nFull conversation history:")
print("=" * 60)
for i, msg in enumerate(chat.history):
    role = msg.role.upper()
    text = msg.parts[0].text[:100].replace("\n", " ")
    print(f"[{i+1}] {role:8}: {text}...")

print(f"\nTotal messages in history: {len(chat.history)}")

---
## Section 6: Model Parameter Optimization

Different tasks need different settings. Let's explore the key parameters.

| Parameter | Range | Best For |
|-----------|-------|---------|
| `temperature` | 0.0 - 1.0 | Low = deterministic, High = creative |
| `max_output_tokens` | 1 - 8192 | Controls response length |
| `top_p` | 0.0 - 1.0 | Nucleus sampling diversity |
| `top_k` | 1 - 40 | Vocabulary diversity |

In [ ]:
# GenerationConfig — all key parameters
generation_configs = {
    "Creative Writing":  genai.types.GenerationConfig(temperature=1.0, max_output_tokens=512, top_p=0.95),
    "Factual QA":        genai.types.GenerationConfig(temperature=0.1, max_output_tokens=256, top_p=0.8),
    "Balanced":          genai.types.GenerationConfig(temperature=0.7, max_output_tokens=512, top_p=0.9),
    "Code Generation":   genai.types.GenerationConfig(temperature=0.2, max_output_tokens=1024, top_p=0.8),
}

prompt = "Describe the colour blue."

print("Testing different temperature settings")
print("=" * 60)
for config_name, config in generation_configs.items():
    response = model.generate_content(prompt, generation_config=config)
    preview = response.text[:180].replace("\n", " ")
    print(f"\nConfig: {config_name} (temp={config.temperature})")
    print(f"  {preview}...")

In [ ]:
# Token counting BEFORE making an API call
text_to_count = "Tell me about the history of artificial intelligence in great detail."

token_count = model.count_tokens(text_to_count)
print(f"Token count for input text:")
print(f"  Text  : {text_to_count[:60]}...")
print(f"  Tokens: {token_count.total_tokens}")

# Estimate cost before calling
PRICING = {
    "gemini-1.5-flash": {"input": 0.075, "output": 0.30},
    "gemini-1.5-pro":   {"input": 3.50,  "output": 10.50},
}

def estimate_cost(model_name, prompt_tokens, expected_output_tokens=200):
    """Estimate API call cost in USD."""
    p = PRICING.get(model_name, {"input": 1.0, "output": 3.0})
    input_cost  = (prompt_tokens / 1_000_000) * p["input"]
    output_cost = (expected_output_tokens / 1_000_000) * p["output"]
    return input_cost + output_cost

cost = estimate_cost("gemini-1.5-flash", token_count.total_tokens)
print(f"  Est. cost: ${cost:.8f}")

---
## Section 7: Structured JSON Responses

Force the model to return structured data you can parse in code.

In [ ]:
import json

def extract_json(text: str):
    """Extract JSON from model response (handles markdown code fences)."""
    # Strip markdown code fences
    fence = re.search(r'```(?:json)?\s*([\s\S]*?)```', text, re.IGNORECASE)
    if fence:
        try:
            return json.loads(fence.group(1).strip())
        except:
            pass
    # Try raw JSON object
    for pattern in [r'(\{[\s\S]*\})', r'(\[[\s\S]*\])']:
        match = re.search(pattern, text)
        if match:
            try:
                return json.loads(match.group(1))
            except:
                continue
    return None

# Example 1: Sentiment analysis
sentiment_prompt = """
Analyse the sentiment of this review:
'This AI chatbot is absolutely incredible! It understands context perfectly.'

Return ONLY a JSON object with these exact fields:
{
  "sentiment": "positive or neutral or negative",
  "confidence": number between 0.0 and 1.0,
  "emotions": ["list", "of", "emotions"],
  "reason": "one-sentence explanation"
}
"""

response = model.generate_content(sentiment_prompt)
data = extract_json(response.text)

print("Sentiment Analysis Result:")
print(json.dumps(data, indent=2))

In [ ]:
# Example 2: Entity extraction
entity_prompt = """
Extract named entities from this text:
'Apple CEO Tim Cook announced a new partnership with OpenAI in San Francisco on Monday.'

Return ONLY a JSON object with these fields:
{
  "persons": [],
  "organisations": [],
  "locations": [],
  "dates": [],
  "other": []
}
"""

response = model.generate_content(entity_prompt)
entities = extract_json(response.text)

print("Entity Extraction Result:")
if entities:
    for key, values in entities.items():
        if values:
            print(f"  {key.title()}: {values}")

In [ ]:
# Example 3: Structured code review
code_to_review = '''
def calculate_average(nums):
    total = 0
    for n in nums:
        total = total + n
    avg = total / len(nums)
    return avg
'''

review_prompt = f"""
Review this Python code and return a JSON object:
{{
  "overall_rating": "number 1-10",
  "issues": [{{"line": "line_number", "issue": "description", "severity": "low|medium|high"}}],
  "improvements": ["list of improvements"],
  "summary": "one-sentence summary"
}}

Code:
{code_to_review}
"""

response = model.generate_content(review_prompt)
review = extract_json(response.text)

if review:
    print(f"Code Review - Rating: {review.get('overall_rating', 'N/A')}/10")
    print(f"Issues found: {len(review.get('issues', []))}")
    for issue in review.get('issues', []):
        print(f"  [{str(issue.get('severity', '?')).upper()}] {issue.get('issue', '')}")
    print(f"Summary: {review.get('summary', 'N/A')}")

---
## Section 8: Robust Error Handling

Production apps need to handle errors gracefully. Let's implement retry logic and error classification.

In [ ]:
import time

class GeminiError(Exception):
    """Custom exception for Gemini API errors."""
    def __init__(self, message, error_type="unknown", retryable=False):
        super().__init__(message)
        self.error_type = error_type
        self.retryable  = retryable

def classify_error(exception) -> GeminiError:
    """Convert raw exceptions to typed GeminiErrors."""
    msg = str(exception).lower()
    if "quota" in msg or "rate" in msg or "429" in msg:
        return GeminiError(str(exception), "rate_limit", retryable=True)
    if "api key" in msg or "auth" in msg or "401" in msg or "403" in msg:
        return GeminiError(str(exception), "authentication", retryable=False)
    if "network" in msg or "connection" in msg or "503" in msg:
        return GeminiError(str(exception), "network", retryable=True)
    if "safety" in msg or "blocked" in msg:
        return GeminiError(str(exception), "safety_filter", retryable=False)
    return GeminiError(str(exception), "unknown", retryable=True)

def call_with_retry(prompt: str, max_retries: int = 3, base_delay: float = 1.5):
    """
    Call the Gemini API with exponential back-off retry logic.

    Parameters
    ----------
    prompt      : str   - The prompt to send
    max_retries : int   - Maximum number of retry attempts
    base_delay  : float - Initial delay in seconds (doubles each retry)

    Returns
    -------
    str - The model response text
    """
    last_error = None

    for attempt in range(1, max_retries + 1):
        try:
            print(f"  Attempt {attempt}/{max_retries}...")
            response = model.generate_content(prompt)
            print(f"  Success on attempt {attempt}!")
            return response.text

        except Exception as e:
            last_error = classify_error(e)

            if not last_error.retryable:
                print(f"  Non-retryable error ({last_error.error_type}): {e}")
                raise last_error

            if attempt < max_retries:
                delay = base_delay * (2 ** (attempt - 1))
                print(f"  {last_error.error_type} - retrying in {delay:.1f}s...")
                time.sleep(delay)
            else:
                print(f"  All {max_retries} retries failed.")
                raise last_error

# Test it
print("Testing robust API call with error handling:")
result = call_with_retry("What is the capital of France? Answer in one word.")
print(f"\nResult: {result.strip()}")

In [ ]:
# Safety filter handling
def safe_generate(prompt: str, fallback: str = "I cannot respond to that request.") -> str:
    """Generate content with safety filter handling."""
    try:
        response = model.generate_content(prompt)
        if not response.candidates:
            return fallback
        candidate = response.candidates[0]
        if str(candidate.finish_reason) == "SAFETY":
            reasons = [r.category.name for r in candidate.safety_ratings
                       if r.probability.name in ("HIGH", "MEDIUM")]
            return f"Content filtered: {reasons}"
        return response.text
    except Exception as e:
        err = classify_error(e)
        if err.error_type == "safety_filter":
            return fallback
        raise

# Test safe generation
result = safe_generate("Write a haiku about autumn leaves.")
print("Safe generation test:")
print(result)

---
## Section 9: API Cost Management

Track and estimate your API costs to stay within budget.

In [ ]:
class CostTracker:
    """Track token usage and estimate API costs across multiple calls."""

    PRICING = {
        "gemini-1.5-flash": {"input": 0.075, "output": 0.30},
        "gemini-1.5-pro":   {"input": 3.50,  "output": 10.50},
        "gemini-1.0-pro":   {"input": 0.50,  "output": 1.50},
    }

    def __init__(self, model_name: str = "gemini-1.5-flash", budget_usd: float = None):
        self.model_name   = model_name
        self.budget_usd   = budget_usd
        self.calls        = []
        self.total_prompt = 0
        self.total_output = 0
        self.total_cost   = 0.0

    def record(self, response, label: str = ""):
        """Record token usage from an API response."""
        usage   = response.usage_metadata
        prompt  = usage.prompt_token_count or 0
        output  = usage.candidates_token_count or 0
        pricing = self.PRICING.get(self.model_name, {"input": 1.0, "output": 3.0})
        cost    = (prompt / 1e6) * pricing["input"] + (output / 1e6) * pricing["output"]

        self.calls.append({"label": label, "prompt": prompt, "output": output, "cost": cost})
        self.total_prompt += prompt
        self.total_output += output
        self.total_cost   += cost

        if self.budget_usd and self.total_cost > self.budget_usd:
            print(f"Budget exceeded! ${self.total_cost:.6f} / ${self.budget_usd:.2f}")
        return cost

    def summary(self):
        print("\n" + "="*50)
        print("API Cost & Usage Summary")
        print("="*50)
        print(f"  Total calls   : {len(self.calls)}")
        print(f"  Prompt tokens : {self.total_prompt:,}")
        print(f"  Output tokens : {self.total_output:,}")
        print(f"  Total tokens  : {self.total_prompt + self.total_output:,}")
        print(f"  Est. cost     : ${self.total_cost:.8f}")
        if self.budget_usd:
            remaining = max(self.budget_usd - self.total_cost, 0)
            print(f"  Budget left   : ${remaining:.6f}")
        print("="*50)

# Usage demonstration
tracker = CostTracker("gemini-1.5-flash", budget_usd=0.01)

prompts = [
    ("What is Python?", "Q1"),
    ("What is machine learning?", "Q2"),
    ("What is a neural network?", "Q3"),
]

print("Making tracked API calls...")
for prompt_text, label in prompts:
    resp = model.generate_content(prompt_text)
    cost = tracker.record(resp, label=label)
    print(f"  [{label}] Cost: ${cost:.8f}")

tracker.summary()

---
## Section 10: Building the Complete AI Chat Application

Combining everything into a production-ready chat class.

In [ ]:
class AIChatApplication:
    """
    A complete, production-ready AI Chat Application.

    Features:
    - Multi-turn conversation with persistent history
    - Custom system instructions
    - Model parameter control
    - Token tracking and cost estimation
    - Robust error handling with retry logic
    - Conversation save/load
    """

    def __init__(
        self,
        api_key: str,
        model_name: str = "gemini-1.5-flash",
        system_prompt: str = "You are a helpful AI assistant.",
        temperature: float = 0.7,
        max_tokens: int = 2048,
        max_history: int = 20,
        budget_usd: float = None,
    ):
        genai.configure(api_key=api_key)
        self.model_name    = model_name
        self.system_prompt = system_prompt
        self.temperature   = temperature
        self.max_tokens    = max_tokens
        self.max_history   = max_history
        self.history       = []
        self.tracker       = CostTracker(model_name, budget_usd=budget_usd)
        self._init_model()
        print(f"AIChatApplication ready (model: {model_name})")

    def _init_model(self):
        self.model = genai.GenerativeModel(
            model_name=self.model_name,
            system_instruction=self.system_prompt,
        )
        self.gen_config = genai.types.GenerationConfig(
            temperature=self.temperature,
            max_output_tokens=self.max_tokens,
        )

    def chat(self, user_message: str) -> str:
        """Send a message and get a response."""
        self.history.append({"role": "user", "parts": [{"text": user_message}]})

        if len(self.history) > self.max_history:
            self.history = self.history[-self.max_history:]

        try:
            chat_session = self.model.start_chat(history=self.history[:-1])
            response     = chat_session.send_message(
                user_message,
                generation_config=self.gen_config,
            )
            assistant_text = response.text
            self.history.append({"role": "model", "parts": [{"text": assistant_text}]})
            self.tracker.record(response)
            return assistant_text

        except Exception as e:
            error = classify_error(e)
            if error.retryable:
                time.sleep(2)
                return self.chat(user_message)
            return f"Error: {error}"

    def set_system_prompt(self, prompt: str):
        """Update the system instruction."""
        self.system_prompt = prompt
        self._init_model()
        print("System prompt updated.")

    def clear_history(self):
        """Clear conversation history (system prompt preserved)."""
        self.history = []
        print("Conversation history cleared.")

    def save_conversation(self, filepath: str):
        """Save the conversation to a JSON file."""
        data = {
            "model": self.model_name,
            "system_prompt": self.system_prompt,
            "timestamp": datetime.now().isoformat(),
            "messages": self.history,
        }
        with open(filepath, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=2, ensure_ascii=False)
        print(f"Saved to: {filepath}")

    def load_conversation(self, filepath: str):
        """Load a previously saved conversation."""
        with open(filepath, "r", encoding="utf-8") as f:
            data = json.load(f)
        self.history = data.get("messages", [])
        print(f"Loaded {len(self.history)} messages from: {filepath}")

    def get_summary(self) -> str:
        """Return conversation summary stats."""
        turns = sum(1 for m in self.history if m["role"] == "user")
        return (
            f"Model: {self.model_name} | "
            f"Turns: {turns} | "
            f"Messages: {len(self.history)} | "
            f"Est. cost: ${self.tracker.total_cost:.6f}"
        )


# Demo: Run the complete chat application
print("AIChatApplication Demo")
print("=" * 60)

app = AIChatApplication(
    api_key=GEMINI_API_KEY,
    model_name="gemini-1.5-flash",
    system_prompt=(
        "You are a concise AI tutor specialising in Python programming. "
        "Always include a short code example in your answers."
    ),
    temperature=0.7,
    max_tokens=1024,
    budget_usd=0.05,
)

# Simulate a multi-turn conversation
conversation = [
    "What is a Python decorator?",
    "Can you show me a decorator that times how long a function takes?",
    "How do I apply multiple decorators to one function?",
]

for user_msg in conversation:
    print(f"\nUser: {user_msg}")
    ai_reply = app.chat(user_msg)
    print(f"AI  : {ai_reply[:500]}...")
    print(f"Stats: {app.get_summary()}")

In [ ]:
# Final summary and cost report
app.tracker.summary()

# Save the conversation
app.save_conversation("my_conversation.json")

print("\nModule 3 Complete!")
print("="*60)
print("You have learned:")
items = [
    "LLM API authentication with environment variables",
    "Connecting to Google Gemini API",
    "System, user, and assistant message roles",
    "Multi-turn conversation & chat history",
    "Model parameter optimization (temperature, max_tokens)",
    "Structured JSON responses",
    "Robust error handling with retry logic",
    "Token usage tracking and cost estimation",
    "Building a complete AI Chat Application class",
]
for item in items:
    print(f"  [x] {item}")